# Point 2 — Visible Bias Initialization

This notebook investigates the **initialization of visible biases** proposed
by Hinton (Section 8 of *A Practical Guide to Training RBMs*, 2010).

The idea: instead of initializing the visible biases to zero, set each bias
$a_i$ to the log-odds of the pixel frequency in the training set:

$$a_i = \log\!\frac{p_i}{1 - p_i}$$

where $p_i$ is the fraction of training images in which pixel $i$ is on.
This gives the model an immediate head start by encoding the **marginal
statistics** of the data into the visible layer *before* any learning
takes place.

We compare training with zero biases vs Hinton-initialized biases across
reconstruction error, pseudo-likelihood, free-energy gap, AIS
log-likelihood, and visual reconstruction quality.

> Run cells **top to bottom**. The notebook is self-contained.

## 0. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 19
})

from scipy.special import logit
from sklearn.datasets import fetch_openml

from rbm_pro_max import RBM

## 1. Data Loading and Preprocessing

We use digits **0, 1, 2** from MNIST, normalized to `[0, 1]` and clipped
(no Gaussian noise, to keep the analysis of bias initialization clean).

In [ ]:
X_original, Y_original = fetch_openml("mnist_784", version=1,
                                      return_X_y=True, as_frame=False)

X_original = np.array(X_original, dtype=np.float32)
Y_original = np.array(Y_original, dtype=int)

# Filter digits 0, 1, 2
mask = (Y_original == 0) | (Y_original == 1) | (Y_original == 2)
X_filtered = X_original[mask]
Y_filtered = Y_original[mask]

# Normalize to [0, 1]
X_train = np.clip(X_filtered.astype(np.float32) / 255.0, 0, 1)

n_v  = X_train.shape[1]      # 784
side = int(np.sqrt(n_v))     # 28

print(f"Filtered 0,1,2 : {X_filtered.shape}")
print(f"Range : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Mean  : {X_train.mean():.3f}")

## 2. Empirical Pixel Probabilities and Hinton Biases

We compute $p_i = \langle v_i \rangle_{\text{data}}$ (clamped to avoid
$\log(0)$) and the corresponding log-odds biases.

In [ ]:
# Empirical pixel probabilities
p_empirical = np.clip(X_train.mean(axis=0), 0.001, 0.999)

# Hinton biases: logit(p) = log(p / (1 - p))
bias_hinton = logit(p_empirical)

# --- Histograms ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(p_empirical, bins=50, color="#2266CC", alpha=0.8)
axes[0].set(title="Empirical Pixel Probabilities $p_i$",
            xlabel="$p_i$", ylabel="Count")
axes[1].hist(bias_hinton, bins=50, color="#CC4422", alpha=0.8)
axes[1].set(title=r"Hinton Visible Biases $a_i = \mathrm{logit}(p_i)$",
            xlabel="Bias value", ylabel="Count")
plt.tight_layout()
plt.show()

## 3. Visual Comparison: Zero vs Hinton Bias

Hinton biases encode the average digit image (darker pixels → more negative
bias → low prior probability). Zero biases carry no prior information.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(X_train.mean(axis=0).reshape(side, side), cmap="gray")
axes[0].set_title("Empirical Mean Image")
axes[0].axis("off")

axes[1].imshow(np.zeros(n_v).reshape(side, side), cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("Zero Initial Bias")
axes[1].axis("off")

im = axes[2].imshow(bias_hinton.reshape(side, side), cmap="jet")
axes[2].set_title("Hinton Bias (Logit)")
axes[2].axis("off")
fig.colorbar(im, ax=axes[2], label="Bias value")

plt.tight_layout()
plt.show()

## 4. Impact on Pre-training Sampling

Before any learning, we compare a single Gibbs step (random input →
hidden → visible) for an RBM with zero biases vs one with Hinton biases.
The Hinton-initialized model already produces digit-like silhouettes,
while the zero-bias model outputs uniform noise.

In [ ]:
np.random.seed(42)

rbm_zeros_pre  = RBM(n_v=n_v, n_h=64, seed=42)
rbm_hinton_pre = RBM(n_v=n_v, n_h=64, seed=42)
rbm_hinton_pre.v_bias = bias_hinton.copy()

v_init = np.random.rand(5, n_v)

_, s_zeros  = rbm_zeros_pre.sample_v(rbm_zeros_pre.sample_h(v_init)[1])
_, s_hinton = rbm_hinton_pre.sample_v(rbm_hinton_pre.sample_h(v_init)[1])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(s_zeros[i].reshape(side, side), cmap="gray")
    axes[0, i].axis("off")
    if i == 2:
        axes[0, i].set_title("Zero Bias — one Gibbs step\n")

    axes[1, i].imshow(s_hinton[i].reshape(side, side), cmap="gray")
    axes[1, i].axis("off")
    if i == 2:
        axes[1, i].set_title("Hinton Bias — one Gibbs step\n")

plt.tight_layout()
plt.show()

## 5. Training Comparison: Zero vs Hinton Init

We train two RBMs with **identical architecture and hyperparameters**
(SGD, lr = 0.005, L2 weight decay = 0.001, w_init = 0.001, fixed batch
size 32, 30 epochs) — the only difference is the visible bias initialization.

Conservative hyperparameters were chosen to keep both models stable and
make the effect of bias initialization clearly visible.

In [ ]:
# --- Model creation ---
rbm_zeros = RBM(
    n_v=n_v, n_h=64, optimizer="sgd", lr=0.005,
    weight_decay=0.001, w_init_scale=0.001, seed=42,
)
rbm_hinton = RBM(
    n_v=n_v, n_h=64, optimizer="sgd", lr=0.005,
    weight_decay=0.001, w_init_scale=0.001, seed=42,
)

# --- Training ---
print("Training RBM with zero biases ...")
res_zeros = rbm_zeros.train(
    X_train, Nepoch=30, Nmini=30, N_ini=32, N_fin=32,
    hinton_init=False, metrics_every=1, verbose=True,
)

print("\nTraining RBM with Hinton bias initialization ...")
res_hinton = rbm_hinton.train(
    X_train, Nepoch=30, Nmini=30, N_ini=32, N_fin=32,
    hinton_init=True, metrics_every=1, verbose=True,
)

print("\nDone.")

## 6. Log-Likelihood via Annealed Importance Sampling

AIS gives a principled estimate of the log-likelihood by estimating the
partition function (Hinton, Section 14; Salakhutdinov & Murray 2008).

In [ ]:
print("Computing log-likelihood via AIS ...")

print("  Estimating ln(Z) for the zero-bias model ...")
ln_Z_zeros  = rbm_zeros.annealed_importance_sampling(n_runs=100, n_betas=1000)
ll_zeros    = rbm_zeros.log_likelihood_ais(X_train, ln_Z=ln_Z_zeros)

print("  Estimating ln(Z) for the Hinton-initialized model ...")
ln_Z_hinton = rbm_hinton.annealed_importance_sampling(n_runs=100, n_betas=1000)
ll_hinton   = rbm_hinton.log_likelihood_ais(X_train, ln_Z=ln_Z_hinton)

print(f"\n[AIS] Log-Likelihood  (zero biases) : {ll_zeros:.4f}")
print(f"[AIS] Log-Likelihood  (Hinton init) : {ll_hinton:.4f}")

## 7. Training Metrics Comparison

Side-by-side evolution of reconstruction error (MSE), pseudo-likelihood,
and free-energy gap for both models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Reconstruction Error ---
ax = axes[0]
ax.plot(res_zeros["metrics"]["epochs"],
        res_zeros["metrics"]["reconstruction_error"],
        "r--o", label="Zero Biases")
ax.plot(res_hinton["metrics"]["epochs"],
        res_hinton["metrics"]["reconstruction_error"],
        "b-s", label="Hinton Init")
ax.set(xlabel="Epoch", ylabel="MSE", title="Reconstruction Error")
ax.grid(True, ls="--", alpha=0.5); ax.legend()

# --- Pseudo-Likelihood ---
ax = axes[1]
ax.plot(res_zeros["metrics"]["epochs"],
        res_zeros["metrics"]["pseudo_likelihood"],
        "r--o", label="Zero Biases")
ax.plot(res_hinton["metrics"]["epochs"],
        res_hinton["metrics"]["pseudo_likelihood"],
        "b-s", label="Hinton Init")
ax.set(xlabel="Epoch", ylabel="log PL", title="Pseudo-Likelihood")
ax.grid(True, ls="--", alpha=0.5); ax.legend()

# --- Free-Energy Gap ---
ax = axes[2]
ax.plot(res_zeros["metrics"]["epochs"],
        res_zeros["metrics"]["free_energy_gap"],
        "r--o", label="Zero Biases")
ax.plot(res_hinton["metrics"]["epochs"],
        res_hinton["metrics"]["free_energy_gap"],
        "b-s", label="Hinton Init")
ax.axhline(0, color="black", ls=":", alpha=0.6)
ax.set(xlabel="Epoch", ylabel="F(data) - F(fantasy)",
       title="Free-Energy Gap")
ax.grid(True, ls="--", alpha=0.5); ax.legend()

plt.tight_layout()
plt.show()

## 8. Visual Reconstruction Quality

Five random digits reconstructed via a single mean-field step
(visible → hidden probabilities → visible probabilities).

In [ ]:
np.random.seed(9)
sample_idx = np.random.choice(X_train.shape[0], 5, replace=False)
test_samples = X_train[sample_idx]

def get_reconstruction(model, X):
    h_prob, _ = model.sample_h(X)
    v_recon, _ = model.sample_v(h_prob)
    return v_recon

recon_zeros  = get_reconstruction(rbm_zeros,  test_samples)
recon_hinton = get_reconstruction(rbm_hinton, test_samples)

fig, axes = plt.subplots(3, 5, figsize=(13, 8))
for i in range(5):
    axes[0, i].imshow(test_samples[i].reshape(side, side), cmap="gray")
    axes[0, i].axis("off")
    if i == 2:
        axes[0, i].set_title("Original Images\n", fontsize=12)

    axes[1, i].imshow(recon_zeros[i].reshape(side, side), cmap="gray")
    axes[1, i].axis("off")
    if i == 2:
        axes[1, i].set_title("Reconstruction (Zero Biases)\n", fontsize=12)

    axes[2, i].imshow(recon_hinton[i].reshape(side, side), cmap="gray")
    axes[2, i].axis("off")
    if i == 2:
        axes[2, i].set_title("Reconstruction (Hinton Init)\n", fontsize=12)

plt.tight_layout()
plt.show()